# Assignment 3

## Learn an XOR Neural Network using gradient-based optimization

Author: Samuel Fredric Berg

Student ID: sb224sc

Date: 2026-04-19

Course: Deep Machine Learning 4DT908

### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Function definitions

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def sigmoid_derivative(x):
    sx = sigmoid(x)
    return sx * (1 - sx)


def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def mse_derivative(y_true, y_pred):
    # Derivative of MSE averaged over N samples: 2/N * (y_pred - y_true)
    n = len(y_true)
    return (2 / n) * (y_pred - y_true)

### Init training data

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = np.array([[0], [1], [1], [0]])

### Init random weights & train network

In [ ]:
np.random.seed(42)

input_size = 2
hidden_size = 5
output_size = 1

weights_1 = 0.1 * np.random.randn(input_size, hidden_size)
biases_1 = np.zeros((1, hidden_size))
weights_2 = 0.1 * np.random.randn(hidden_size, output_size)
biases_2 = np.zeros((1, output_size))

learning_rate = 3.0
epochs = 5000
losses = []

for epoch_idx in range(epochs):
    # Forward pass
    z_1 = X @ weights_1 + biases_1
    a_1 = sigmoid(z_1)
    z_2 = a_1 @ weights_2 + biases_2
    a_2 = sigmoid(z_2)

    loss_val = mse(Y, a_2)
    losses.append(loss_val)

    if epoch_idx % 1000 == 0 or epoch_idx == epochs - 1:
        print(f"Epoch {epoch_idx}, Loss: {loss_val:.5f}")

    # Backward pass (backpropagation)
    # Output layer: dL/dz_2 = mse_derivative * sigmoid_derivative
    dz_2 = mse_derivative(Y, a_2) * sigmoid_derivative(z_2)
    dw_2 = a_1.T @ dz_2
    db_2 = np.sum(dz_2, axis=0, keepdims=True)

    # Hidden layer: backpropagate delta through weights_2
    dz_1 = dz_2 @ weights_2.T * sigmoid_derivative(z_1)
    dw_1 = X.T @ dz_1
    db_1 = np.sum(dz_1, axis=0, keepdims=True)

    # Update weights
    weights_2 -= learning_rate * dw_2
    biases_2 -= learning_rate * db_2
    weights_1 -= learning_rate * dw_1
    biases_1 -= learning_rate * db_1

### Evaluate network

In [ ]:
print("\nFinal outputs after training:")
z1 = X @ weights_1 + biases_1
a1 = sigmoid(z1)
z2 = a1 @ weights_2 + biases_2
a2 = sigmoid(z2)
print(np.round(a2))

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

w_range = np.linspace(weights_2[0, 0] - 1.5, weights_2[0, 0] + 1.5, 50)
v_range = np.linspace(weights_2[1, 0] - 1.5, weights_2[1, 0] + 1.5, 50)
w0, w1 = np.meshgrid(w_range, v_range)
loss = np.zeros_like(w0)

for i in range(w0.shape[0]):
    for j in range(w0.shape[1]):
        W2_tmp = np.copy(weights_2)
        W2_tmp[0, 0] = w0[i, j]
        W2_tmp[1, 0] = w1[i, j]

        a1_tmp = sigmoid(X @ weights_1 + biases_1)
        a2_tmp = sigmoid(a1_tmp @ W2_tmp + biases_2)
        loss[i, j] = mse(Y, a2_tmp)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(w0, w1, loss, cmap="viridis")
ax.set_xlabel("w2[0,0]")
ax.set_ylabel("w2[1,0]")
ax.set_zlabel("Loss")
plt.show()

### Bonus: Verify gradients with PyTorch autodiff

To verify the manually computed gradients, we use PyTorch's automatic differentiation and compare the results with a single forward+backward pass.

In [ ]:
import torch

# Convert data to PyTorch tensors
X_t = torch.tensor(X, dtype=torch.float64)
Y_t = torch.tensor(Y, dtype=torch.float64)

# Use the final trained weights from manual backprop
W1_t = torch.tensor(weights_1, dtype=torch.float64, requires_grad=True)
b1_t = torch.tensor(biases_1, dtype=torch.float64, requires_grad=True)
W2_t = torch.tensor(weights_2, dtype=torch.float64, requires_grad=True)
b2_t = torch.tensor(biases_2, dtype=torch.float64, requires_grad=True)

# Forward pass using PyTorch (sigmoid and MSE)
z1_t = X_t @ W1_t + b1_t
a1_t = torch.sigmoid(z1_t)
z2_t = a1_t @ W2_t + b2_t
a2_t = torch.sigmoid(z2_t)
loss_t = torch.mean((Y_t - a2_t) ** 2)

# Backward pass using PyTorch autodiff
loss_t.backward()

# Manually compute gradients at the same weights
z_1_v = X @ weights_1 + biases_1
a_1_v = sigmoid(z_1_v)
z_2_v = a_1_v @ weights_2 + biases_2
a_2_v = sigmoid(z_2_v)

dz_2_v = mse_derivative(Y, a_2_v) * sigmoid_derivative(z_2_v)
dw_2_manual = a_1_v.T @ dz_2_v
db_2_manual = np.sum(dz_2_v, axis=0, keepdims=True)
dz_1_v = dz_2_v @ weights_2.T * sigmoid_derivative(z_1_v)
dw_1_manual = X.T @ dz_1_v
db_1_manual = np.sum(dz_1_v, axis=0, keepdims=True)

# Compare gradients
print("Gradient verification (manual vs. PyTorch autodiff):")
print(f"  dW2 max abs diff: {np.max(np.abs(dw_2_manual - W2_t.grad.numpy())):.2e}")
print(f"  db2 max abs diff: {np.max(np.abs(db_2_manual - b2_t.grad.numpy())):.2e}")
print(f"  dW1 max abs diff: {np.max(np.abs(dw_1_manual - W1_t.grad.numpy())):.2e}")
print(f"  db1 max abs diff: {np.max(np.abs(db_1_manual - b1_t.grad.numpy())):.2e}")
print("\nGradients match PyTorch autodiff!" if all([
    np.max(np.abs(dw_2_manual - W2_t.grad.numpy())) < 1e-10,
    np.max(np.abs(db_2_manual - b2_t.grad.numpy())) < 1e-10,
    np.max(np.abs(dw_1_manual - W1_t.grad.numpy())) < 1e-10,
    np.max(np.abs(db_1_manual - b1_t.grad.numpy())) < 1e-10,
]) else "\nWARNING: Gradient mismatch detected!")

### Conclusion
The network successfully learned XOR 

In the first plot we can see how the network progressed over time. Given the chosen seed, it learned very little in the first 3000 epochs and then suddenly got down to almost 0 loss after that. This could indicate that the network randomly initialized to a relatively stable region before it got unstable and suddenly dropped down to a different stable region. The second plot shows how sensitive the network is to changes in two weights where the valley is visible. A smooth gradient shows that it is easier for the network to converge.